# HuskySwap teacher: gpt-oss-120b on your own GPU

Serves **openai/gpt-oss-120b** with vLLM (an OpenAI-compatible server, the same API the agent already calls) and exposes it through a free Cloudflare tunnel. The agent then uses it instead of Groq: no daily caps, no rate-limit waits.

**Runtime:** Runtime → Change runtime type → **G4** (RTX PRO 6000, 96 GB). The model needs about 65 GB; L4 and 40 GB A100 are too small.

**Cost:** G4 time uses Colab compute units. Stop the runtime when you're done (Runtime → Disconnect and delete runtime).

**Security:** the tunnel address is public, so the server only answers requests carrying the random API key generated below. Put the key in `.env.local` yourself; don't paste it into chats.

Run the cells in order. Keep this tab open while the agent uses the teacher: Colab disconnects idle or long sessions, and the tunnel address changes every time it restarts.

In [ ]:
# 1. Check the GPU: expect an RTX PRO 6000 with ~97,000 MiB
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install vLLM (a few minutes). Blackwell workstation GPUs need a recent release.
!pip install -q --upgrade vllm
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)"

In [ ]:
# 3. Start the server in the background. The first start downloads ~65 GB (5-15 minutes).
import secrets, subprocess, time, urllib.request

API_KEY = secrets.token_urlsafe(24)  # only requests with this key are answered
MODEL = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL,
     "--host", "127.0.0.1", "--port", "8000",
     "--api-key", API_KEY,
     "--max-model-len", "32768",
     "--gpu-memory-utilization", "0.92",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

start = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; read /content/vllm.log:\n" + open("/content/vllm.log").read()[-3000:])
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5)
        break
    except Exception:
        time.sleep(15)
        print(f"loading... {int(time.time() - start)} s", end="\r")
print(f"\nserver ready after {int(time.time() - start)} s")

In [ ]:
# 4. Open a Cloudflare quick tunnel (free, no account) and print the settings for .env.local
import re, socket
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found.group(0)
        break
if url:
    # A new quick-tunnel address takes a few seconds to a minute to appear in DNS
    host = url.removeprefix("https://")
    for _ in range(45):
        try:
            socket.gethostbyname(host)
            break
        except socket.gaierror:
            time.sleep(2)
    else:
        print("warning: the tunnel address does not resolve yet; wait a minute before using it")
if not url:
    raise RuntimeError("No tunnel address yet; read /content/tunnel.log")
print("Put these in .env.local (keep the Groq lines commented out to switch back):\n")
print(f"TEACHER_BASE_URL={url}/v1")
print(f"TEACHER_API_KEY={API_KEY}")
print(f"TEACHER_MODEL={MODEL}")
print("\nand remove or comment out TEACHER_API_KEYS (the Groq key pool).")

In [ ]:
# 5. Test a tool call through the tunnel, the way the agent calls the model
import json, urllib.error
body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "How many listings are there? Use the tool."}],
    "tools": [{"type": "function", "function": {"name": "run_sql", "description": "Run a SQL query",
               "parameters": {"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]}}}],
    "tool_choice": "auto", "reasoning_effort": "medium", "max_tokens": 1024,
}
req = urllib.request.Request(f"{url}/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"content-type": "application/json", "authorization": f"Bearer {API_KEY}"})
t0 = time.time()
for attempt in range(12):  # the tunnel can take a minute to become reachable
    try:
        reply = json.load(urllib.request.urlopen(req, timeout=120))
        break
    except urllib.error.URLError as e:
        print(f"not reachable yet ({e.reason}); retrying...")
        time.sleep(5)
else:
    raise RuntimeError("tunnel still unreachable; read the tunnel log")
msg = reply["choices"][0]["message"]
print(f"{time.time() - t0:.1f} s | tool calls: {json.dumps(msg.get('tool_calls'))[:300]}")
print("usage:", reply.get("usage"))

In [ ]:
# 6. If something looks wrong: the last lines of the server log
print(open("/content/vllm.log").read()[-4000:])